# 🔥 MODO BESTIA EN KAGGLE: Subagente 4 - Entrevistas Callejeras (Street Interview / Vox Pop)
### Plataforma: Kaggle Dual GPU (T4 x2) | Modelo: Meta-Llama-3.1-8B-Instruct | LoRA r=32

Este cuaderno entrena al especialista en **entrevistas espontáneas en la calle (Vox Pop)**:
1. **Óptica 28mm f/5.6:** Profundidad de campo realista con fondo urbano reconocible y transeúntes consistentes.
2. **Microfonía Real:** Micrófono de mano con cubo de flag apuntado a 15 cm de la boca del entrevistado.
3. **Turnos de Habla Limpios:** Cero solapamiento de voces; secuencia orgánica: sorpresa -> reflexión -> respuesta.
4. **Física del Sujeto:** Mirada al entrevistador fuera de cámara, no al lente publicitario.

> **Importante:** Recuerda tener **Internet ON** y **GPU T4 x2** activados en el panel derecho de Kaggle.

## Paso 1: Instalación de Unsloth y Librerías de Optimización

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar Meta-Llama-3.1-8B-Instruct en 4-Bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Meta-Llama-3.1-8B cargado exitosamente en Kaggle.")

## Paso 3: Configurar LoRA de Alta Capacidad (r = 32, alpha = 64)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ Adaptadores LoRA r=32 configurados para el Especialista Street Interview.")

## Paso 4: Auto-Generar el Dataset Street Interview en Memoria (Cero archivos faltantes)

In [ ]:
import json
import os
from datasets import Dataset

STREET_EXPANDIDO = [
    ("Joven profesional en calle peatonal concurrida", "¿Qué harías si un robot te contesta el WhatsApp?", "Mientras me conteste rápido… ¡mejor que mi ex!"),
    ("Estudiante universitario frente a cafetería urbana", "¿Cuánto dinero consideras suficiente para ser feliz?", "El suficiente para no tener que mirar los precios en el supermercado."),
    ("Turista en avenida comercial de tiendas", "¿Qué le dirías a tu versión de hace diez años?", "Que compre Bitcoin y no se preocupe tanto por el qué dirán de los demás."),
    ("Empresario casual en distrito financiero", "¿Cuál es el peor consejo de finanzas que te han dado?", "Que compres casa propia a los veinte años endeudándote con el banco a treinta años."),
    ("Chica con café caminando por un parque", "¿Qué habilidad consideras imprescindible hoy en día?", "Aprender a adaptarse rápido, saber vender y comunicarse con claridad absoluta."),
    ("Arquitecto frente a rascacielos contemporáneo", "¿Cuál es el edificio más impresionante que has visto?", "La Sagrada Familia en Barcelona, la luz que entra por los vitrales no es de este mundo."),
    ("Chico en patineta en plaza pública", "¿Cuál es tu mayor meta para antes de los treinta?", "Construir algo propio que me dé libertad de tiempo antes que cualquier lujo material."),
    ("Emprendedora gastronómica en mercado gourmet", "¿El secreto para que un negocio prospere?", "Consistencia brutal en la calidad. El cliente siempre nota cuando bajas el estándar.")
]

pares_street = []
for _ in range(5):
    for idea, preg, resp in STREET_EXPANDIDO:
        chosen = (
            f"Maintain the exact subject, casual clothing, the solid mic flag, handheld microphone, interviewer's hand position, golden-hour light and street background from the first frame. The microphone stays firmly pointed at the subject's mouth.\n\n"
            f"CAMERA: Documentary handheld 28mm at eye level, natural operator body sway and small inertial reframes; no cuts, no gimbal smoothness. Background pedestrians walk naturally with consistent shapes and realistic speeds.\n\n"
            f"SUBJECT: First, off-screen interviewer asks clearly: \"{preg}\" During the question, the subject listens with lips sealed, eyebrows popping up in genuine surprise, then glances up-lateral for one second, thinking. Laughs briefly and answers to the interviewer off-camera, not into the lens: \"{resp}\" Gestures loosely with free hand, then shoots one quick playful glance into lens at the end. Only one person speaks at a time; lips close fully on m, b, and p.\n\n"
            f"ENVIRONMENT: Lively pedestrian shopping street at golden hour, warm low sun rim light on hair, soft-focus passersby, urban street energy.\n\n"
            f"Audio: two distinct voices captured by handheld dynamic mic — off-screen interviewer's question, then subject's laughing answer in neutral Spanish; street ambience with footsteps and city murmur, no music, no subtitles."
        )
        rejected = (
            f"Maintain subject. "
            f"CAMERA: Tripod locked static shot. "
            f"SUBJECT: Both interviewer and subject speak at the same time with voice overlap, microphone is pointed at the camera instead of the mouth, subject stares unblinkingly at the lens like a commercial, crowd morphs in background. "
            f"ENVIRONMENT: Artificial studio lights on street. Audio: loud electronic music."
        )
        pares_street.append({
            "prompt": f"Generate an Image-to-Video (I2V) prompt for Veo 3.1 based on this anchor: {idea}. Action: Street interview reaction and answer",
            "chosen": chosen,
            "rejected": rejected
        })

archivo_dpo = "dataset_dpo_street.jsonl"
with open(archivo_dpo, "w", encoding="utf-8") as f:
    for item in pares_street:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

dataset = Dataset.from_list(pares_street)
print(f"✅ Dataset Street auto-generado: {len(dataset)} pares DPO listos en /kaggle/working/.")

## Paso 5: Entrenamiento DPO en Kaggle (GPU T4 x2)

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir="./resultados_street",
    beta=0.15,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    warmup_ratio=0.1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    save_strategy="no",
    seed=3407,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=2048,
    max_prompt_length=512,
)

gpu_mem_used = torch.cuda.memory_allocated() / 1024**3
print(f"📊 VRAM Inicial: {gpu_mem_used:.2f} GB")
print("🔥 Entrenando al Subagente Street Interview en Kaggle... (Duración aprox: 10-12 minutos)")
stats = dpo_trainer.train()
print("🎉 ¡Subagente Street Interview entrenado con éxito!")

## Paso 6: Guardar y Generar Enlaces de Descarga

In [ ]:
import shutil
from IPython.display import FileLink

out_dir = "lora_video_street_8b"
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)

zip_name = f"{out_dir}.zip"
shutil.make_archive(out_dir, 'zip', out_dir)
print(f"📦 Modelo empaquetado como: {zip_name}")

# Enlace HTML de descarga directa
print("\n👉 Haz clic en el siguiente enlace para descargar directamente a tu PC:")
display(FileLink(zip_name))

# Enlace temporal en la nube como respaldo (doble alternativa)
print('\n🌐 Subiendo a servidores temporales de respaldo (copia el link que aparezca abajo):')
!curl -T {zip_name} https://temp.sh/ || true
!curl -F "file=@{zip_name}" https://0x0.st || true
print('\n💡 Recuerda: También puedes descargar el zip desde el panel derecho de Kaggle -> Output -> lora_video_street_8b.zip -> Tres puntos (...) -> Download.')